# FD05: Cost-based threshold policy under an alert budget
Source for the FD05 notebook. Convert with `jupytext --to ipynb` and run it after copying the FD02 export to `data/fd02_scores.csv`, or set `FD02_SCORES_PATH`.
Input: FD02 validation and test scores (uncalibrated XGBoost probabilities). Costs and the 1% alert budget are stated in the README under "Cost assumptions".
Calibration and threshold selection use validation rows only. The test window is reported once.

In [1]:
import json
from pathlib import Path

from fraud_threshold_policy import plots
from fraud_threshold_policy.calibration import calibrated_probabilities, select_calibration
from fraud_threshold_policy.costs import ALERT_BUDGET, FP_COST
from fraud_threshold_policy.data import AMOUNT, SCORE, SPLIT, TARGET, TIME, load_scores, resolve_scores_path
from fraud_threshold_policy.metrics import brier
from fraud_threshold_policy.policy import evaluate_test, select_threshold
from fraud_threshold_policy.windows import WINDOW_SECONDS, alert_rate_by_window

IN_NOTEBOOKS = Path.cwd().name == "notebooks"
FIG_DIR = Path("../reports/figures") if IN_NOTEBOOKS else Path("reports/figures")
REPORT_DIR = Path("../reports") if IN_NOTEBOOKS else Path("reports")
DATASET = "Credit Card Fraud (FD02 export)"
MODEL = "FD02 XGBoost"

In [2]:
df = load_scores(resolve_scores_path())
val = df[df[SPLIT] == "val"]
test = df[df[SPLIT] == "test"]
for name, part in (("val", val), ("test", test)):
    hours = (part[TIME].max() - part[TIME].min()) / 3600
    print(f"{name}: {len(part):,} rows, {int(part[TARGET].sum())} frauds, span {hours:.1f} h")

val: 42,718 rows, 56 frauds, span 5.1 h
test: 42,721 rows, 52 frauds, span 6.0 h


## Cost assumptions (from README, before tuning)

In [3]:
print(f"c_FP per alerted legitimate transaction: {FP_COST}")
print(f"c_FN per missed fraud: its Amount")
print(f"alert budget (share of transactions alerted): {ALERT_BUDGET:.2%}")

c_FP per alerted legitimate transaction: 2.0
c_FN per missed fraud: its Amount
alert budget (share of transactions alerted): 1.00%


## Calibrate on validation only

In [4]:
calibrator, method, val_briers = select_calibration(val[SCORE], val[TARGET])
print(f"validation Brier by method: {val_briers}")
print(f"chosen calibration: {method}")

validation Brier by method: {'isotonic': 0.0002695291158255849, 'sigmoid': 0.0003091506126453293}
chosen calibration: isotonic


## Choose the threshold on validation only

In [5]:
val_probs = calibrated_probabilities(calibrator, val[SCORE])
choice = select_threshold(val[TARGET], val[AMOUNT], val_probs, budget=ALERT_BUDGET, fp_cost=FP_COST)
choice.grid.to_csv(REPORT_DIR / "threshold_selection.csv", index=False)
print(f"chosen threshold: {choice.threshold:.6f}")
print(f"validation expected cost: {choice.expected_cost:,.2f}, alert rate: {choice.alert_rate:.4%}")

chosen threshold: 0.078947
validation expected cost: 234.16, alert rate: 0.2224%


In [6]:
plots.plot_cost_curve(
    choice.grid,
    choice.threshold,
    f"Expected cost vs threshold, {DATASET}, validation split, {MODEL} calibrated ({method})",
    FIG_DIR / "fd05_cost_curve_val.png",
)

PosixPath('../reports/figures/fd05_cost_curve_val.png')

## Test window (reported once)

In [7]:
test_raw = test[SCORE].to_numpy()
test_probs = calibrated_probabilities(calibrator, test_raw)
brier_raw = brier(test[TARGET], test_raw)
brier_cal = brier(test[TARGET], test_probs)
print(f"test Brier, raw score: {brier_raw:.6f}; calibrated ({method}): {brier_cal:.6f}")

results = evaluate_test(test[TARGET], test[AMOUNT], test_probs, choice.threshold, ALERT_BUDGET, FP_COST)
for label, metrics in results.items():
    print(label, metrics)

test Brier, raw score: 0.000453; calibrated (isotonic): 0.000394
policy {'threshold': 0.07894736842105263, 'expected_cost': 2466.4, 'alert_rate': 0.0020130614920062734, 'within_budget': True, 'fp': 47, 'missed_frauds': 13, 'missed_fraud_amount': 2372.4, 'confusion': {'tn': 42622, 'fp': 47, 'fn': 13, 'tp': 39}, 'recall_by_amount': 0.6154245179027634, 'recall_by_count': 0.75}
naive_0_5 {'threshold': 0.5, 'expected_cost': 2387.1600000000003, 'alert_rate': 0.0010533461295381661, 'within_budget': True, 'fp': 7, 'missed_frauds': 14, 'missed_fraud_amount': 2373.1600000000003, 'confusion': {'tn': 42662, 'fp': 7, 'fn': 14, 'tp': 38}, 'recall_by_amount': 0.615301318877981, 'recall_by_count': 0.7307692307692307}


In [8]:
plots.plot_reliability(
    test[TARGET],
    {"raw FD02 score": test_raw, f"calibrated ({method})": test_probs},
    f"Reliability diagram, {DATASET}, test split, {MODEL}",
    FIG_DIR / "fd05_reliability_test.png",
)

PosixPath('../reports/figures/fd05_reliability_test.png')

In [9]:
windows = alert_rate_by_window(test[TIME], test_probs, choice.threshold, WINDOW_SECONDS)
print(windows)
plots.plot_alert_rate_by_window(
    windows,
    ALERT_BUDGET,
    f"Alert rate per 6-hour window, {DATASET}, test split, {MODEL} calibrated ({method})",
    FIG_DIR / "fd05_alert_rate_window_test.png",
)

   window_start  transactions  alerts  alert_rate
0      151200.0         42721      86    0.002013


PosixPath('../reports/figures/fd05_alert_rate_window_test.png')

In [10]:
report = {
    "model": f"{MODEL}, calibrated",
    "calibration_method": method,
    "calibration_validation_brier": val_briers,
    "brier_test_raw": brier_raw,
    "brier_test_calibrated": brier_cal,
    "chosen_threshold": choice.threshold,
    "validation_expected_cost": choice.expected_cost,
    "validation_alert_rate": choice.alert_rate,
    "test": results,
    "cost_assumptions": {
        "fp_cost": FP_COST,
        "missed_fraud_cost": "Amount",
        "alert_budget": ALERT_BUDGET,
    },
}
(REPORT_DIR / "policy_test_report.json").write_text(json.dumps(report, indent=2))
print(f"saved {REPORT_DIR / 'policy_test_report.json'}")

saved ../reports/policy_test_report.json
